In [1]:
!pip install torch_xla

In [2]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.utils.utils as xu
from torch.utils.data import Dataset, DataLoader

/usr/local/lib/python3.11/dist-packages/torch_xla/__init__.py:253: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [3]:
##############################################
# 1. Helper: Split Flat Token List by Composition
##############################################
def split_by_composer(token_list):
    """
    Splits a flat token list into a list of compositions.
    A new composition starts every time a token that is a string starting with
    "COMPOSER_" is encountered.
    """
    compositions = []
    current = []
    for token in token_list:
        # If token is a composer marker, start a new composition
        if isinstance(token, str) and token.startswith("COMPOSER_"):
            if current:
                compositions.append(current)
            current = []
        else:
            current.append(token)
    if current:
        compositions.append(current)
    return compositions

In [4]:
##############################################
# 2. Load Data from JSON Files
##############################################
with open("tokenized_events.json", "r") as f:
    flat_tokenized_events = json.load(f)   # Flat list: mix of composer tokens (strings) and event tokens (ints)

with open("relative_time_compositions.json", "r") as f:
    flat_relative_tokens = json.load(f)      # Flat list: same structure as tokenized_events

# Split the flat lists into compositions using the composer tokens
tokenized_compositions = split_by_composer(flat_tokenized_events)
relative_compositions = split_by_composer(flat_relative_tokens)



In [5]:
 relative_compositions = [sum(seq, []) for seq in relative_compositions]

In [6]:
print(f"Type of relative_compositions: {type(relative_compositions)}")
print(f"Length: {len(relative_compositions)}")

for i, seq in enumerate(relative_compositions[:5]):  # Check first 5
    print(f"Sequence {i}: Type={type(seq)}, First element type={type(seq[0]) if isinstance(seq, list) and seq else 'N/A'}")


Type of relative_compositions: <class 'list'>
Length: 1
Sequence 0: Type=<class 'list'>, First element type=<class 'int'>


In [7]:
##############################################
# 3. Compute Vocabulary Sizes
##############################################
# tokenized_compositions should now be a list of lists of ints.
# (Composer tokens have been removed by the split function.)
# We assume that each token is an int.
if tokenized_compositions and isinstance(tokenized_compositions[0][0], int):
    vocab_size = max([max(seq) for seq in tokenized_compositions]) + 1
else:
    raise ValueError("tokenized_compositions is not a list of int sequences.")

if relative_compositions and isinstance(relative_compositions[0][0], int):
    time_vocab_size = max([max(seq) for seq in relative_compositions]) + 1
else:
    raise ValueError("relative_compositions is not a list of int sequences.")

print(f"Vocab size: {vocab_size}, Time vocab size: {time_vocab_size}")


Vocab size: 2008932, Time vocab size: 13751


In [8]:
##############################################
# 4. Create a Dataset Class
##############################################
class MusicDataset(Dataset):
    def __init__(self, event_data, time_data, seq_length=None):
        """
        event_data: List of compositions, each is a list of event token IDs (ints)
        time_data:  List of compositions, each is a list of relative time token IDs (ints)
        seq_length: If provided, sequences will be padded/truncated to this length.
        """
        assert len(event_data) == len(time_data), "Mismatch in number of compositions."
        self.event_data = event_data
        self.time_data = time_data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.event_data)

    def __getitem__(self, idx):
        events = torch.tensor(self.event_data[idx], dtype=torch.long)
        times = torch.tensor(self.time_data[idx], dtype=torch.long)
        if self.seq_length is not None:
            if events.size(0) < self.seq_length:
                pad_size = self.seq_length - events.size(0)
                events = torch.cat([events, torch.zeros(pad_size, dtype=torch.long)])
                times = torch.cat([times, torch.zeros(pad_size, dtype=torch.long)])
            else:
                events = events[:self.seq_length]
                times = times[:self.seq_length]
        return events, times

In [9]:
##############################################
# 5. Define the Transformer-XL (Decoder-Only) Model
##############################################
class TransformerXLBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

class TransformerXL(nn.Module):
    def __init__(self, vocab_size, time_vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        # Embedding layers for event tokens and relative time tokens
        self.event_embedding = nn.Embedding(vocab_size, d_model)
        self.time_embedding  = nn.Embedding(time_vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerXLBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, event_tokens, time_tokens):
        """
        event_tokens: [batch, seq_len] of event token IDs
        time_tokens:  [batch, seq_len] of relative time token IDs
        """
        event_emb = self.event_embedding(event_tokens)  # [batch, seq_len, d_model]
        time_emb  = self.time_embedding(time_tokens)      # [batch, seq_len, d_model]

        # Combine the event and time embeddings by adding them.
        x = event_emb + time_emb

        for layer in self.layers:
            x = layer(x)

        logits = self.fc_out(x)
        return logits

In [10]:
##############################################
# 6. TPU Setup and DataLoader
##############################################
device = xm.xla_device()

# Hyperparameters (adjust as needed)
seq_length = 256
batch_size = 2  # Reduce to prevent OOM
d_model = 384   # Lower from 512 to save memory
n_layers = 6    # Slightly fewer layers
n_heads = 6     # Reduce attention heads to lower memory usage
d_ff = 1024     # Reduce feed-forward size (large values eat memory)
dropout = 0.1
  # Prevents overfitting


dataset = MusicDataset(tokenized_compositions, relative_compositions, seq_length=seq_length)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model on TPU
model = TransformerXL(vocab_size, time_vocab_size, d_model, n_layers, n_heads, d_ff, dropout).to(device, dtype=torch.bfloat16)



In [11]:
##############################################
# 7. Training Loop (Autoregressive Decoder)
##############################################
criterion = nn.CrossEntropyLoss().to(device, dtype=torch.bfloat16)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
num_epochs = 50

def train_step(model, event_data, time_data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    # Teacher forcing: input is sequence[:-1], target is sequence[1:]
    inputs_events, targets_events = event_data[:, :-1], event_data[:, 1:]
    inputs_times = time_data[:, :-1]

    logits = model(inputs_events, inputs_times)  # [batch, seq_len-1, vocab_size]
    loss = criterion(logits.view(-1, vocab_size), targets_events.view(-1))

    loss.backward()
    xm.optimizer_step(optimizer)
    xm.mark_step()
    return loss.item()

for epoch in range(num_epochs):
    total_loss = 0
    for batch in data_loader:
        event_batch, time_batch = batch
        event_batch = event_batch.to(device)
        time_batch = time_batch.to(device)
        loss = train_step(model, event_batch, time_batch, optimizer, criterion)
        total_loss += loss
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(data_loader):.4f}")

# Save the model state_dict
torch.save(model.state_dict(), "transformer_xl_music_tpu.pth")
print("Model saved as transformer_xl_music_tpu.pth")

Epoch 1/50 | Loss: 14.6875
Epoch 2/50 | Loss: 14.0625
Epoch 3/50 | Loss: 13.7500
Epoch 4/50 | Loss: 13.5000
Epoch 5/50 | Loss: 13.1875
Epoch 6/50 | Loss: 12.8125
Epoch 7/50 | Loss: 12.3750
Epoch 8/50 | Loss: 11.9375
Epoch 9/50 | Loss: 11.5000
Epoch 10/50 | Loss: 11.1250
Epoch 11/50 | Loss: 10.7500
Epoch 12/50 | Loss: 10.3125
Epoch 13/50 | Loss: 9.9375
Epoch 14/50 | Loss: 9.5000
Epoch 15/50 | Loss: 9.1250
Epoch 16/50 | Loss: 8.7500
Epoch 17/50 | Loss: 8.3750
Epoch 18/50 | Loss: 8.0000
Epoch 19/50 | Loss: 7.6562
Epoch 20/50 | Loss: 7.3438
Epoch 21/50 | Loss: 7.0312
Epoch 22/50 | Loss: 6.7500
Epoch 23/50 | Loss: 6.4688
Epoch 24/50 | Loss: 6.1875
Epoch 25/50 | Loss: 5.9375
Epoch 26/50 | Loss: 5.6562
Epoch 27/50 | Loss: 5.4375
Epoch 28/50 | Loss: 5.1875
Epoch 29/50 | Loss: 4.9688
Epoch 30/50 | Loss: 4.7500
Epoch 31/50 | Loss: 4.5312
Epoch 32/50 | Loss: 4.3125
Epoch 33/50 | Loss: 4.1250
Epoch 34/50 | Loss: 3.8906
Epoch 35/50 | Loss: 3.6875
Epoch 36/50 | Loss: 3.5000
Epoch 37/50 | Loss: 3.328